
Goal: embed table + column metadata into ChromaDB so the agent retrieves
only the relevant schema context per question instead of dumping everything
into the prompt. Only needed once your dataset has many columns.


In [1]:
import sqlite3
from fastembed import TextEmbedding
import chromadb
import numpy as np


DB_PATH    = 'data.db'
TABLE      = 'data'
COLLECTION = 'schema_store'

c:\Users\Ashish\anaconda3\envs\talktocsv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#Step 1 — extract column metadata from SQLite

In [2]:
conn    = sqlite3.connect(DB_PATH)
cols    = conn.execute(f'PRAGMA table_info({TABLE})').fetchall()
samples = conn.execute(f'SELECT * FROM {TABLE} LIMIT 5').fetchall()
conn.close()

col_names = [c[1] for c in cols]

# build one document per column — this is what gets embedded
documents = []
for i, c in enumerate(cols):
    col_name    = c[1]
    col_type    = c[2]
    sample_vals = [str(row[i]) for row in samples if row[i] is not None][:3]
    sample_str  = ", ".join(sample_vals)
    doc         = f"Table: {TABLE} | Column: {col_name} | Type: {col_type} | Sample values: {sample_str}"
    documents.append(doc)

print(f"Built {len(documents)} column documents")
for d in documents:
    print(d)

Built 17 column documents
Table: data | Column: job_title_short | Type: TEXT | Sample values: Data Analyst, Data Analyst, Data Analyst
Table: data | Column: job_title | Type: TEXT | Sample values: Summer Internship -Data Analyst Intern, Risk Management, Staff Data Analyst Operations, Infrastructure & Systems, Junior Data Analyst - Entry Level
Table: data | Column: job_location | Type: TEXT | Sample values: Marlborough, MA, Fremont, CA, Waco, TX
Table: data | Column: job_via | Type: TEXT | Sample values: via Boatingrevealed.com, via ClimateTechList, via ZipRecruiter
Table: data | Column: job_schedule_type | Type: TEXT | Sample values: Full-time, Part-time, and Internship, Full-time, Full-time and Part-time
Table: data | Column: job_work_from_home | Type: INTEGER | Sample values: 0, 0, 0
Table: data | Column: search_location | Type: TEXT | Sample values: New York, United States, California, United States, Texas, United States
Table: data | Column: job_posted_date | Type: TEXT | Sample va

## Step 2 — embed and store in ChromaDB

In [3]:
chroma = chromadb.Client()
embed_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# delete collection if it already exists
try:
    chroma.delete_collection(COLLECTION)
except:
    pass

# create collection WITHOUT embedding_function — we handle embeddings manually
collection = chroma.create_collection(COLLECTION)

# embed documents using fastembed
embeddings = [e.tolist() for e in embed_model.embed(documents)]

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"col_{i}" for i in range(len(documents))],
)
print(f"Stored {collection.count()} embeddings")

Stored 17 embeddings


## Step 3 — retrieve relevant columns for a question

In [ ]:
def get_relevant_schema(question: str, n_results: int = 5) -> str:
    q_embedding = list(embed_model.embed([question]))[0].tolist()
    results     = collection.query(
        query_embeddings=[q_embedding],
        n_results=n_results
    )
    docs = results["documents"][0]
    return f"Table: {TABLE}\nRelevant columns:\n" + "\n".join(f"  - {d}" for d in docs)

# test retrieval
test_questions = [
    "What is the average yearly salary by job title?",
    "Which columns have missing values?",
    "What are the top skills required for Data Engineer?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(get_relevant_schema(q))

Q: What is the average yearly salary by job title?
Table: data
Relevant columns:
  - Table: data | Column: salary_year_avg | Type: REAL | Sample values: 
  - Table: data | Column: salary_rate | Type: TEXT | Sample values: 
  - Table: data | Column: salary_hour_avg | Type: REAL | Sample values: 
  - Table: data | Column: job_title_short | Type: TEXT | Sample values: Data Analyst, Data Analyst, Data Analyst
  - Table: data | Column: job_title | Type: TEXT | Sample values: Summer Internship -Data Analyst Intern, Risk Management, Staff Data Analyst Operations, Infrastructure & Systems, Junior Data Analyst - Entry Level
--------------------------------------------------
Q: Which columns have missing values?
Table: data
Relevant columns:
  - Table: data | Column: job_no_degree_mention | Type: INTEGER | Sample values: 0, 1, 1
  - Table: data | Column: job_title_short | Type: TEXT | Sample values: Data Analyst, Data Analyst, Data Analyst
  - Table: data | Column: job_health_insurance | Type: I

## Step 4 — confirm it helps vs full schema

In [5]:
# full schema token count (rough)
conn       = sqlite3.connect(DB_PATH)
cols_raw   = conn.execute(f'PRAGMA table_info({TABLE})').fetchall()
conn.close()
full_schema = 'Table: ' + TABLE + '\n' + '\n'.join(f'  - {c[1]} ({c[2]})' for c in cols_raw)

print('Full schema chars :', len(full_schema))
print('RAG schema chars  :', len(get_relevant_schema('total revenue by region')))
print()
print('RAG pays off once you have 20+ columns — for small datasets, full schema is fine')

Full schema chars : 462
RAG schema chars  : 536

RAG pays off once you have 20+ columns — for small datasets, full schema is fine
